In [1]:
!pip install python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 17.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
from pptx import Presentation
from pptx.util import Inches, Pt

# 1. 挂载 Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- 配置路径 ---
# 输出文件将直接保存在 MyDrive 根目录下
output_pptx_path = '/content/drive/MyDrive/Diffusion_new.pptx'

# 数据读取路径 (保持您之前的设置，如有变动请修改这里)
base_data_path = '/content/drive/MyDrive/Path_squared_new/'

samples_list = range(5)
num_nodes_list = [20, 30, 50, 80, 100]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 初始化 PPT
prs = Presentation()

print(f"开始生成 PPT，目标路径: {output_pptx_path}")

for num_nodes in num_nodes_list:
    for sample in samples_list:
        print(f"正在处理: Nodes {num_nodes}, Sample {sample}")

        # 构造读取文件的完整路径
        data_file = os.path.join(base_data_path, f'data_{sample}_{num_nodes}.pt')
        model_file = os.path.join(base_data_path, f'checkpoint_d{num_nodes}_exp{sample}.pth')

        # --- 1. 加载 Ground Truth ---
        try:
            data_all = torch.load(data_file, weights_only=False)
            graph_adj = data_all['graph_adj'] # 真实图 (0/1)
        except Exception as e:
            print(f"  [跳过] 无法加载数据 {data_file}: {e}")
            continue

        # --- 2. 加载模型并获取原始概率 (No Threshold) ---
        try:
            checkpoint = torch.load(model_file, map_location=device)
            state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

            if 'att_mask_pro' in state_dict:
                att_mask_logits = state_dict['att_mask_pro']
                # Softmax 归一化
                probs = torch.softmax(att_mask_logits, dim=-1)

                # 获取 "存在边" 的概率 (索引1)，保持浮点数，不进行二值化
                adj_estimated = probs[:, :, 1].detach().cpu().numpy()
            else:
                print("  [警告] 模型中未找到 att_mask_pro")
                adj_estimated = np.zeros((num_nodes, num_nodes))

        except Exception as e:
            print(f"  [错误] 无法加载模型 {model_file}: {e}")
            adj_estimated = np.zeros((num_nodes, num_nodes))

        # --- 3. 创建 PPT 幻灯片 ---
        slide = prs.slides.add_slide(prs.slide_layouts[6]) # 空白版式

        # 添加标题
        title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(9), Inches(0.5))
        tf = title_box.text_frame
        tf.text = f"Nodes: {num_nodes} | Sample: {sample}"
        tf.paragraphs[0].font.size = Pt(24)
        tf.paragraphs[0].font.bold = True

        # --- 绘图 A: Ground Truth (左侧) ---
        fig1 = plt.figure(figsize=(5, 4))
        sns.heatmap(graph_adj, cbar=False, cmap='Blues', xticklabels=False, yticklabels=False)
        plt.title("Ground Truth")

        # 保存到内存流
        stream1 = io.BytesIO()
        plt.savefig(stream1, format='png', bbox_inches='tight', dpi=100)
        stream1.seek(0)
        plt.close(fig1)

        # --- 绘图 B: Diffusion Output Raw (右侧) ---
        fig2 = plt.figure(figsize=(5.5, 4)) # 略宽以容纳 colorbar
        # vmin=0, vmax=1 确保颜色深浅代表绝对概率
        sns.heatmap(adj_estimated, cbar=True, cmap='Reds', vmin=0, vmax=1, xticklabels=False, yticklabels=False)
        plt.title("Diffusion Probabilities (Raw)")

        # 保存到内存流
        stream2 = io.BytesIO()
        plt.savefig(stream2, format='png', bbox_inches='tight', dpi=100)
        stream2.seek(0)
        plt.close(fig2)

        # --- 插入图片到 PPT ---
        # 左图位置
        slide.shapes.add_picture(stream1, Inches(0.5), Inches(1.5), height=Inches(4.5))
        # 右图位置
        slide.shapes.add_picture(stream2, Inches(5.0), Inches(1.5), height=Inches(4.5))

# --- 4. 保存文件 ---
prs.save(output_pptx_path)
print("-" * 30)
print(f"成功! 文件已保存到: {output_pptx_path}")
print("请去 Google Drive 根目录查看 'Diffusion_origin.pptx'")

Mounted at /content/drive
开始生成 PPT，目标路径: /content/drive/MyDrive/Diffusion_new.pptx
正在处理: Nodes 20, Sample 0
正在处理: Nodes 20, Sample 1
正在处理: Nodes 20, Sample 2
正在处理: Nodes 20, Sample 3
正在处理: Nodes 20, Sample 4
正在处理: Nodes 30, Sample 0
正在处理: Nodes 30, Sample 1
正在处理: Nodes 30, Sample 2
正在处理: Nodes 30, Sample 3
正在处理: Nodes 30, Sample 4
正在处理: Nodes 50, Sample 0
正在处理: Nodes 50, Sample 1
正在处理: Nodes 50, Sample 2
正在处理: Nodes 50, Sample 3
正在处理: Nodes 50, Sample 4
正在处理: Nodes 80, Sample 0
  [跳过] 无法加载数据 /content/drive/MyDrive/Path_squared_new/data_0_80.pt: [Errno 2] No such file or directory: '/content/drive/MyDrive/Path_squared_new/data_0_80.pt'
正在处理: Nodes 80, Sample 1
  [跳过] 无法加载数据 /content/drive/MyDrive/Path_squared_new/data_1_80.pt: [Errno 2] No such file or directory: '/content/drive/MyDrive/Path_squared_new/data_1_80.pt'
正在处理: Nodes 80, Sample 2
  [跳过] 无法加载数据 /content/drive/MyDrive/Path_squared_new/data_2_80.pt: [Errno 2] No such file or directory: '/content/drive/MyDrive/Path_squared_new/d